In [ ]:
import kagglehub
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import warnings
warnings.filterwarnings('ignore')


# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:

# Load the CSV file
pokemon_path = os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(pokemon_path)
print(f"Shape: {df.shape}")


In [ ]:
# Task 2: Write your code here:

df.head(5)

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
#  distribution
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=30, edgecolor='black')
plt.title('Delivery_Time Distribution')
plt.xlabel('Delivery_Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df_Drop = df.drop(columns=['Order_ID'])
df_Drop.head()

In [ ]:
# Task 2: Write your code here:


# 2. Do we have missing values?
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")



#Handling Missing Target Column:
df_clean = df_Drop.dropna(subset=['Delivery_Time'])



#Handiling Catagorical Columns:
categorical_cols = df_clean.select_dtypes(include=["object"]).columns

df_clean[categorical_cols]= df_clean[categorical_cols].fillna('none')

#Handiling Numirical Columns:
number_cols = df_clean.select_dtypes(include=["number"]).columns

df_clean[number_cols] = df[number_cols].fillna(df[number_cols].mean())



df_clean.head()











In [ ]:
# Task 3: Write your code here:

#Do we have duplicate samples?
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)




In [ ]:
# Task 4: Write your code here:

print('data before encoding:\n', categorical_cols) #show before encoding
le = LabelEncoder()

for col in categorical_cols:
  df_clean[col] = le.fit_transform(df_clean[col].astype(str))


df_clean.head()


In [ ]:
# Task 5: Write your code here:


scaler = StandardScaler()
Scaled_DF = scaler.fit_transform(df_clean)
Scaled_DF




In [ ]:
# Task 6: Write your code here:
import seaborn as sns

# 1. Is the target imbalanced?
def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()
check_target_imbalance(df_clean, "Delivery_Time")

#Balanced




In [ ]:
# Task1 : Write your code here:
target_column = "Delivery_Time"
X = df_clean.drop(target_column, axis=1)
y = df_clean[target_column]

y.head()

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, cross_val_score
from sklearn.ensemble import RandomForestRegressor





kf = KFold(n_splits=5, shuffle=True, random_state=42)

model = RandomForestRegressor(n_estimators=100, random_state=42)

# 4. Train and Evaluate

scores = cross_val_score(model, X, y, cv=kf, scoring='neg_mean_absolute_error')

# Convert negative scores to positive MAE
mae_scores = -scores

# 5. Print the averaged score
print(f"Average MAE across folds: {np.mean(mae_scores):.4f}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_predict

# Note: Ideally, use your 'X' and 'y' data here.
# Since X and y were not present in the context, I generated synthetic data for demonstration.
# ... (Synthetic data generation skipped for brevity, refer to full tool code if needed) ...

# Assuming X and y are defined:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
model = RandomForestRegressor(n_estimators=100, random_state=42)

# 1. Fit the model to get Feature Importance
model.fit(X, y)

# Plot Feature Importance
importances = model.feature_importances_
indices = np.argsort(importances)[::-1]
feature_names = X.columns

plt.figure(figsize=(10, 6))
plt.title("Feature Importance")
plt.bar(range(X.shape[1]), importances[indices], align="center")
plt.xticks(range(X.shape[1]), feature_names[indices], rotation=45)
plt.tight_layout()
plt.show()



In [ ]:
# Task 2: Write your code here:

predicted_y = cross_val_predict(model, X, y, cv=kf)

# Plot Histogram
plt.figure(figsize=(10, 6))
plt.hist(predicted_y, bins=30, edgecolor='k', alpha=0.7)
plt.title("Histogram of Predicted Delivery Times")
plt.xlabel("Predicted Time")
plt.ylabel("Frequency")
plt.show()

In [ ]:
# Task Bonus: Write your code here:
!pip install catboost

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from catboost import CatBoostRegressor

# 1. Setup KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# List to store the MAE of the averaged predictions for each fold
ensemble_mae_scores = []

print("Starting Cross-Validation Loop...\n")

# 2. Manual Loop
for fold, (train_index, val_index) in enumerate(kf.split(X, y)):


    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]


    rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
    rf_model.fit(X_train, y_train)
    rf_pred = rf_model.predict(X_val)


    cb_model = CatBoostRegressor(verbose=0, random_state=42)
    cb_model.fit(X_train, y_train)
    cb_pred = cb_model.predict(X_val)


    avg_pred = (rf_pred + cb_pred) / 2


    fold_mae = mean_absolute_error(y_val, avg_pred)
    ensemble_mae_scores.append(fold_mae)

    print(f"Fold {fold+1} MAE: {fold_mae:.4f}")

# 3. Print Final Average
print("-" * 30)
print(f"Average Ensemble MAE: {np.mean(ensemble_mae_scores):.4f}")